# STITCHV2 Tutorial Notebook

This notebook mirrors the top-level `TUTORIAL.md` and gives a runnable, notebook-friendly version of the same guidance.

It focuses on:

- exact input file formats
- CLI and interactive Python usage
- STITCH-parity configuration
- the current STITCHV2 command surface
- examples for exports and utility commands

This notebook is instruction-only by default. Cells are written so you can adapt and execute them when you are ready.

## 1. Mental Model

`stitchv2 run` does four main things:

1. reads sample metadata and variant positions
2. extracts read evidence from BAM/CRAM, optionally mixing in founders and microarray data
3. runs the HMM block by block
4. writes chunked parquet outputs, with optional calibration and export

Original STITCH is organized around an R entrypoint and VCF-centric outputs. STITCHV2 is organized around a Python package, parquet-first outputs, explicit batching, mixed ploidy, and optional post-HMM calibration.

## 2. Exact Input File Formats

### 2.1 Samples Table

`--samples` accepts parquet or a delimited text file readable by pandas.

Required columns:

| Column | Type | Meaning |
| --- | --- | --- |
| `sample_id` | string | unique sample name used in outputs |
| `bam_path` | string | BAM/CRAM path; empty string is allowed |
| `generation` | float | generations since founder mosaic start |

Optional columns:

| Column | Type | Meaning |
| --- | --- | --- |
| `sex` | string | used by `--ploidy-males` and `--ploidy-females` |
| `father_id`, `mother_id` | string | pedigree columns when pedigree is embedded in samples |
| `plink_path` | string | per-sample PLINK prefix for array evidence |

Accepted sex labels include `M`, `male`, `1`, `XY`, `F`, `female`, `2`, and `XX`.

Example CSV:

```text
sample_id,bam_path,generation,sex,father_id,mother_id
s1,/data/bams/s1.bam,100,F,,
s2,/data/bams/s2.bam,100,M,,
child1,/data/bams/child1.bam,101,F,s2,s1
```

### 2.2 Positions Table

`--positions` accepts parquet or delimited text.

Required columns:

| Column | Type | Meaning |
| --- | --- | --- |
| `CHR` | string | chromosome or contig name |
| `POS` | integer | 1-based variant position |
| `REF` | string | reference allele |
| `ALT` | string | alternate allele |

Example text:

```text
CHR POS REF ALT
chr19 3000001 A G
chr19 3000104 C T
```

### 2.3 Founder Inputs

STITCHV2 can use:

| Source | How to provide it | When to use it |
| --- | --- | --- |
| Uniform mutable founders | no founder input | exploratory runs |
| Founder VCF | `--founder-vcf founders.vcf.gz` | parity or known-founder runs |
| Founder PLINK | `--founder-plink founders_prefix` | founders already in BED/BIM/FAM |
| In-memory `FounderPanel` | Python API | tests and custom workflows |

`--founder-immutable` freezes supplied founders.

### 2.4 Pedigree Input

`--pedigree` accepts parquet or delimited text. Default columns are `sample_id`, `father_id`, and `mother_id`.

### 2.5 Microarray Evidence

`--microarray-plink PREFIX` loads hard array genotypes from a PLINK BED/BIM/FAM prefix and injects them as evidence.

In [ ]:
from pathlib import Path

import pandas as pd

samples_example = pd.DataFrame(
    {
        "sample_id": ["s1", "s2", "child1"],
        "bam_path": ["/data/bams/s1.bam", "/data/bams/s2.bam", "/data/bams/child1.bam"],
        "generation": [100.0, 100.0, 101.0],
        "sex": ["F", "M", "F"],
        "father_id": ["", "", "s2"],
        "mother_id": ["", "", "s1"],
    }
)

positions_example = pd.DataFrame(
    {
        "CHR": ["chr19", "chr19"],
        "POS": [3000001, 3000104],
        "REF": ["A", "C"],
        "ALT": ["G", "T"],
    }
)

samples_example

In [ ]:
positions_example

## 3. Converting Classic STITCH Text Inputs

If you already have `bamlist.txt`, `sample_names.txt`, and `pos.txt`, convert them into STITCHV2-style tables like this:

In [ ]:
from pathlib import Path

import pandas as pd

def stitch_text_to_stitchv2_tables(data_dir: Path, generation: float = 10.0):
    bam_paths = pd.read_csv(data_dir / "bamlist.txt", header=None, names=["bam_path"])
    names_path = data_dir / "sample_names.txt"
    if names_path.exists():
        sample_ids = pd.read_csv(names_path, header=None, names=["sample_id"])
    else:
        sample_ids = pd.DataFrame({"sample_id": [f"sample_{i}" for i in range(len(bam_paths))]})
    samples = pd.concat([sample_ids, bam_paths], axis=1)
    samples["generation"] = float(generation)
    positions = pd.read_csv(
        data_dir / "pos.txt",
        sep=r"\s+",
        header=None,
        names=["CHR", "POS", "REF", "ALT"],
    )
    return samples, positions

## 4. Quick CLI Start

Minimal run:

```bash
stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir runs/chr1 \
  --n-founders 8 \
  --em-iterations 5 \
  --block-size 1000 \
  --hmm-backend jax \
  --fragment-coupling-model stitch_parity \
  --write-genotype-posteriors \
  --write-genotype-calls
```

Notebook-friendly shell cell:


In [ ]:
%%bash
echo stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir runs/chr1 \
  --n-founders 8 \
  --em-iterations 5 \
  --block-size 1000 \
  --hmm-backend jax \
  --fragment-coupling-model stitch_parity \
  --write-genotype-posteriors \
  --write-genotype-calls

## 5. STITCH-Parity Runs

For a STITCH-style comparison, the main points are:

- keep `fragment_coupling_model=stitch_parity`
- use enough EM iterations, usually `5-10`
- make the calibration choice explicit
- if founders are known, provide them and freeze them

In STITCHV2, the practical equivalent of "true founders + hard immutable" is:

- provide founders through `--founder-vcf`, `--founder-plink`, or an in-memory `FounderPanel`
- set `--founder-immutable`

There is no literal `--use-true-founders` CLI flag in STITCHV2 at present. The equivalent behavior is to supply the true founder panel as input.

In [ ]:
%%bash
echo stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir runs/parity_raw \
  --n-founders 8 \
  --founder-vcf founders.truth.vcf.gz \
  --founder-immutable \
  --em-iterations 5 \
  --block-size 1000 \
  --hmm-backend jax \
  --fragment-likelihood-mode augment \
  --fragment-coupling-model stitch_parity \
  --genotype-call-mode stitch_no_call \
  --genotype-call-stitch-threshold 0.9 \
  --no-calibrate-genotype-posteriors \
  --write-genotype-posteriors \
  --write-genotype-calls \
  --write-support-mask

In [ ]:
import pandas as pd

from stitchv2 import FounderConfig, PipelineConfig, StitchPipeline

cfg = PipelineConfig(
    chromosome="chr1",
    positions_path="positions.parquet",
    output_dir="runs/parity_python",
    n_founders=8,
    em_iterations=5,
    block_size=1000,
    hmm_backend="jax",
    fragment_likelihood_mode="augment",
    fragment_coupling_model="stitch_parity",
    calibrate_genotype_posteriors=False,
    genotype_call_mode="stitch_no_call",
    genotype_call_stitch_threshold=0.9,
    write_genotype_posteriors=True,
    write_genotype_calls=True,
    write_support_mask=True,
    founder=FounderConfig(
        source_format="vcf",
        source_path="founders.truth.vcf.gz",
        immutable=True,
    ),
)

samples = pd.read_parquet("samples.parquet")
# Remove the leading comment when you are ready to execute.
# StitchPipeline(cfg).prepare_inputs(samples)

## 6. Sex Chromosomes, Mixed Ploidy, and Ploidy-Zero Behavior

Mixed-sex chrX example:

```bash
stitchv2 run \
  --samples samples_with_sex.parquet \
  --positions positions_chrX.parquet \
  --chromosome chrX \
  --output-dir runs/chrX \
  --n-founders 8 \
  --ploidy 2 \
  --ploidy-males 1 \
  --ploidy-females 2 \
  --fragment-coupling-model stitch_parity \
  --write-genotype-posteriors \
  --write-genotype-calls
```

When a sample resolves to ploidy `0`, STITCHV2 keeps that sample in the output but does not include it in the HMM. The outputs stay shape-stable:

- genotype calls are `-1`
- dosages are missing
- haplotype probabilities are zero vectors when that output is requested

## 7. Interactive Python API Example

This example builds a tiny in-memory founder panel and shows the standard Python entrypoint.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from stitchv2 import PipelineConfig, StitchPipeline
from stitchv2.founders import FounderPanel

samples = pd.DataFrame(
    {
        "sample_id": ["s1", "s2"],
        "bam_path": ["/data/s1.bam", "/data/s2.bam"],
        "generation": [100.0, 100.0],
    }
)

positions = pd.DataFrame(
    {
        "CHR": ["chr1", "chr1"],
        "POS": [3000001, 3000104],
        "REF": ["A", "C"],
        "ALT": ["G", "T"],
    }
)

positions_path = Path("positions.parquet")
# positions.to_parquet(positions_path, index=False)

founders = FounderPanel(
    chromosome="chr1",
    positions=positions["POS"].to_numpy(dtype=np.int64),
    ref=positions["REF"].astype(str).to_numpy(),
    alt=positions["ALT"].astype(str).to_numpy(),
    alt_prob=np.full((8, len(positions)), 0.5, dtype=np.float32),
    immutable_mask=np.zeros(8, dtype=bool),
)

cfg = PipelineConfig(
    chromosome="chr1",
    positions_path=positions_path,
    output_dir="runs/interactive",
    n_founders=8,
    block_size=1000,
    em_iterations=2,
    hmm_backend="jax",
    read_mode="read_stream",
    read_stream_backend="auto",
    fragment_likelihood_mode="augment",
    fragment_coupling_model="stitch_parity",
    write_genotype_posteriors=True,
    write_genotype_calls=True,
)

# Remove the leading comments when you are ready to execute.
# positions.to_parquet(positions_path, index=False)
# StitchPipeline(cfg).prepare_inputs(samples, founder_panel=founders)

## 8. Output Files

STITCHV2 writes parquet datasets under `--output-dir`.

| Path | When written | Meaning |
| --- | --- | --- |
| `samples.parquet` | always | normalized sample table used by the run |
| `positions.parquet` | always | filtered positions used by the run |
| `founders.parquet` | always | founder panel used after initialization |
| `dosage/block=*.parquet` | always | per-sample, per-site dosage |
| `recombination/block=*.parquet` | always | per-site recombination summaries |
| `founder_updates/block=*.parquet` | always | founder ALT probabilities |
| `transitions/block=*.parquet` | when enabled | compact transition summaries |
| `haplotype_probabilities/block=*.parquet` | `--write-haplotype-probabilities` | haplotype outputs |
| `genotype_posteriors/block=*.parquet` | `--write-genotype-posteriors` | genotype posterior vectors |
| `genotype_calls/block=*.parquet` | `--write-genotype-calls` | hard genotype calls |
| `support_mask/block=*.parquet` | `--write-support-mask` | direct-evidence mask |
| `stage_timings.json` | always | per-block timing breakdown |
| `memory_profile_summary.json` | unless disabled | memory summary |
| `dask_run_summary.json` | Dask runs | Dask runtime metadata |
| `cli_run_summary.json` | CLI runs | exact CLI state summary |
| `run_summary.json` | CLI runs | stable copy of the run summary |

## 9. Main CLI Parameters

The full authoritative parameter definitions live in the CLI implementation, but these are the categories you will use most often.

### 9.1 Core `stitchv2 run` Parameters

| Category | Parameters |
| --- | --- |
| required inputs | `--samples`, `--positions`, `--chromosome`, `--output-dir`, `--n-founders` |
| region | `--chr-start`, `--chr-end` |
| ploidy | `--ploidy`, `--ploidy-males`, `--ploidy-females` |
| HMM | `--block-size`, `--em-iterations`, `--hmm-backend`, `--jax-sample-batch-size` |
| read extraction | `--read-mode`, `--read-stream-backend`, `--io-workers`, `--htslib-threads-per-file` |
| fragment model | `--fragment-likelihood-mode`, `--fragment-coupling-model`, `--fragment-max-diff-reads`, `--fragment-max-emission-diff` |
| outputs | `--write-transitions`, `--write-haplotype-probabilities`, `--write-genotype-posteriors`, `--write-genotype-calls`, `--write-support-mask` |
| calibration | `--no-calibrate-genotype-posteriors`, `--calibration-mode`, `--genotype-posterior-temperature`, `--genotype-posterior-blend`, `--use-lightgbm-calibrator` |
| call policy | `--genotype-call-mode`, `--genotype-call-min-confidence`, `--genotype-call-min-margin`, `--genotype-call-stitch-threshold` |
| microarray | `--microarray-plink`, `--microarray-generation-default`, `--microarray-hard-call-weight`, `--no-microarray-add-samples` |
| pedigree | `--pedigree`, `--pedigree-mode`, `--pedigree-strength`, `--pedigree-iterations` |
| founders | `--founder-vcf`, `--founder-plink`, `--founder-immutable` |
| Dask | `--executor`, `--dask-n-workers`, `--dask-threads-per-worker`, `--dask-performance-report`, `--dask-task-stream`, `--dask-target-task-memory-mb` |

For the full parameter-by-parameter wording, see the top-level `TUTORIAL.md`.

## 10. Utility Commands

### `stitchv2 cv`

Cross-validation and tuning harness with pseudo-truth input.

```bash
stitchv2 cv \
  --samples samples.parquet \
  --positions positions.parquet \
  --pseudo-truth pseudo_truth.parquet \
  --chromosome chr1 \
  --output-dir runs/cv \
  --k-values 6,8,10 \
  --ngen-values 0.75,1.0,1.25 \
  --s-values 2,3 \
  --seeds 0,1,2 \
  --folds 5 \
  --holdout-fraction 0.2 \
  --hmm-backend jax \
  --fragment-coupling-model stitch_parity
```

### `stitchv2 tune-jax-memory`

Profiles block size, memmap use, and JAX sample batching.

### `stitchv2 combine`

Combines partitioned parquet chunks after a run.

### `stitchv2 export-vcf` and `stitchv2 export-bcf`

Exports parquet outputs into VCF or BCF.

### `stitchv2 reformat-stitch-filenames`

Reformats STITCH-oriented filenames into STITCHV2 naming and writes a JSON mapping.

In [ ]:
%%bash
echo stitchv2 export-vcf \
  --run-output-dir runs/production \
  --output-vcf runs/production/production.vcf.gz \
  --chromosome chr1 \
  --threads 4

In [ ]:
%%bash
echo stitchv2 combine \
  --run-output-dir runs/production \
  --datasets all \
  --output-dir runs/production/combined

## 11. Practical Parameter Choices

The parameters that usually matter first are:

1. `--n-founders`
2. `--em-iterations`
3. `--block-size`
4. `--jax-sample-batch-size`
5. `--fragment-likelihood-mode` and `--fragment-coupling-model`
6. calibration and hard-call policy

Rules of thumb:

- use `hmm_backend=jax` for serious larger runs
- start with `block-size=1000` unless memory says otherwise
- use `em-iterations=5` for parity and benchmark runs
- leave calibration on by default unless the goal is raw model comparison
- use `genotype-call-mode=stitch_no_call` when you want STITCH-like no-call behavior

## 12. Troubleshooting and Preflight Checklist

| Symptom | Likely cause | What to check |
| --- | --- | --- |
| no reads overlap variants | contig mismatch or wrong coordinates | compare BAM headers, `--chromosome`, and `positions.CHR` |
| Dask is slower than serial | tasks are too small or the job is too small | increase block size or sample batching |
| JAX OOM | too many samples, SNPs, or founders per task | lower `--block-size` or `--jax-sample-batch-size` |
| STITCHV2 differs from STITCH | founder behavior, calibration, or call policy differs | use immutable supplied founders, `stitch_parity`, and comparable call settings |
| sex chromosome output looks wrong | `sex` labels are missing or inconsistent | normalize `sex` and set male and female ploidy explicitly |

Before a large run, confirm:

- `generation` values are correct
- contig names match across BAMs, positions, and founders
- founder behavior is intentional
- parity runs use `fragment_coupling_model=stitch_parity`
- the calibration choice is explicit
- the requested outputs cover what you need downstream